In [1]:
import pandas as pd


sms_db = pd.read_csv("./all_sms.csv")
sms_db['date'] = pd.to_datetime(sms_db['date'], unit='s')
sms_db.dtypes

id                     int64
date          datetime64[us]
sender                   str
text                     str
is_from_me              bool
service                  str
dtype: object

In [2]:
sms_db = sms_db[sms_db['sender'] != 'me']
sms_db

,id,date,sender,text,is_from_me,service
2,3,2021-05-21 07:07:00.000000,VD-HDFCBK,Your mobile number and device is successfully ...,False,SMS
3,4,2021-05-21 07:07:03.000000,VK-HDFCBK,Your mobile number and device is successfully ...,False,SMS
5,6,2021-05-21 07:09:37.000000,VK-HDFCBK,Your mobile number and device is successfully ...,False,SMS
7,8,2021-05-21 07:09:40.000000,VK-HDFCBK,Device SMS count Exceeded For The Day,False,SMS
10,11,2021-05-21 08:19:47.000000,VK-ICICIB,"Dear Customer, registration for UPI has starte...",False,SMS
...,...,...,...,...,...,...
12124,18204,2026-03-04 14:26:16.869241,TRAIND-G,"ವಂಚಕರ ವಿರುದ್ಧ ಸ್ಪ್ಯಾಮ್ ವರದಿ ಮಾಡಿ. ಆದರೆ, ಫೋನಿನ ...",False,SMS
12125,18212,2026-03-05 07:24:09.798212,TRAIND-G,स्पॅमरच्या विरोधात कारवाईसाठी स्पॅमचा अहवाल द्...,False,SMS
12126,18215,2026-03-05 08:04:54.587784,AIRTEL-S,Never respond to emails/embedded links in mess...,False,SMS
12127,18223,2026-03-05 22:34:08.041849,MAHABK-S,"Dear Customer, there is no transaction in your...",False,SMS


In [3]:
# Check whether `id` is sequential and identify gaps/duplicates
id_series = pd.to_numeric(sms_db["id"], errors="coerce").dropna().astype(int)

row_count = len(sms_db)
min_id = id_series.min()
max_id = id_series.max()
unique_ids = id_series.nunique()

is_exact_1_to_n = (min_id == 1) and (max_id == row_count) and (unique_ids == row_count)
is_contiguous_min_to_max = (unique_ids == (max_id - min_id + 1))

print(f"rows: {row_count}")
print(f"id min/max: {min_id}/{max_id}")
print(f"unique ids: {unique_ids}")
print(f"sequential 1..rows? {is_exact_1_to_n}")
print(f"contiguous min..max? {is_contiguous_min_to_max}")

# Missing IDs in the observed min..max range
expected_ids = pd.Index(range(min_id, max_id + 1))
missing_ids = expected_ids.difference(pd.Index(id_series.unique()))

print(f"missing id count: {len(missing_ids)}")
print("first 20 missing ids:", missing_ids[:20].tolist())

# Duplicate IDs (if any)
dup_counts = id_series.value_counts()
duplicate_ids = dup_counts[dup_counts > 1]
print(f"duplicate id count: {len(duplicate_ids)}")
if len(duplicate_ids) > 0:
    print("first 20 duplicate ids:", duplicate_ids.head(20).to_dict())

rows: 11984
id min/max: 3/18252
unique ids: 11984
sequential 1..rows? False
contiguous min..max? False
missing id count: 6266
first 20 missing ids: [5, 7, 9, 10, 14, 15, 26, 35, 36, 37, 38, 39, 40, 50, 60, 66, 79, 84, 99, 137]
duplicate id count: 0


In [4]:
sms_db['sender'].value_counts()

sender
VZ-611123          1007
VZ-612345          1001
VD-611126           256
AD-HDFCBK           242
JZ-JioPay           207
                   ... 
ADHAAR-S(smsft)       1
+918977520171         1
HDFCBN-S(smsft)       1
SBIBNK-S(smsft)       1
AIRTEL-S              1
Name: count, Length: 1648, dtype: int64

In [5]:
# Filter out sent messages
inbox_df = sms_db[sms_db['sender'] != 'me'].copy()

# Basic sender stats
unique_senders = inbox_df['sender'].nunique()
print(f"Total incoming messages: {len(inbox_df)}")
print(f"Total unique senders: {unique_senders}")

# Top 20 senders by volume
top_senders = inbox_df['sender'].value_counts().head(50)
print("\nTop 20 senders by message volume:")
print(top_senders)

Total incoming messages: 11984
Total unique senders: 1648

Top 20 senders by message volume:
sender
VZ-611123    1007
VZ-612345    1001
VD-611126     256
AD-HDFCBK     242
JZ-JioPay     207
VM-611121     192
VM-HDFCBK     163
VZ-EXPIRY     146
JK-620016     146
JD-620016     136
JM-620016     134
VD-HDFCBN     133
JZ-JIOINF     132
JZ-JIOPAY     125
JX-HDFCBK     121
JX-620016     119
VM-HDFCBN     116
VM-ViCARE     114
VD-HDFCBK     111
VM-611126     109
VM-IDFCFB     101
5512111       100
TX-SLCEIT      96
VZ-ViRoam      81
QP-CHGGIn      71
JZ-JioSvc      70
VK-NSETRA      66
VM-NSESMS      63
CP-OneCrd      62
AX-HDFCBK      59
BW-SBIUPI      58
AD-HDFCBN      57
VM-FLPKRT      56
TM-IDFCFB      56
5040406        55
TM-TATANU      55
JM-HDFCBN      54
JZ-JIOVOC      53
VM-adidas      52
50404          51
BH-CHGGIn      49
BA-SBIUPI      49
VD-IDFCFB      49
JD-SBIUPI      47
VM-BSELTD      45
AD-OneCrd      43
AD-IDFCFB      43
AD-SBIUPI      40
VM-NOBRKR      40
VK-MAHABK      39


In [6]:
import re

def categorize_sender(sender):
    sender = str(sender).strip()
    
    # 1. Regular Mobile Number: optional '+' followed by 10 to 15 digits
    if re.match(r'^\+?[0-9]{10,15}$', sender):
        return 'Mobile Number'
    
    # 2. Numeric Shortcode: purely numeric, less than 10 digits
    elif re.match(r'^[0-9]{3,9}$', sender):
        return 'Numeric Shortcode'
    
    # 3. Commercial/Brand: standard XX-YYYYYY Indian gateway format
    elif re.match(r'^[A-Za-z]{2}-.+$', sender):
        return 'Commercial/Brand (Prefixed)'
    
    # 4. Commercial/Brand: other textual names (e.g., 'JioPay', 'HDFCBK' without prefix)
    elif re.search(r'[A-Za-z]', sender):
        return 'Commercial/Brand (Textual)'
    
    # 5. Others: to catch edge cases
    else:
        return 'Others'

# Apply the categorization
inbox_df['sender_category'] = inbox_df['sender'].apply(categorize_sender)

# Get counts
category_counts = inbox_df['sender_category'].value_counts()
print("Message volume by Sender Category:")
print(category_counts, "\n")

# Let's inspect unique senders in other/edge-case categories
others_senders = inbox_df[inbox_df['sender_category'] == 'Others']['sender'].unique()
print(f"Unique senders in 'Others': {len(others_senders)}")
if len(others_senders) > 0:
    print(others_senders[:20])

Message volume by Sender Category:
sender_category
Commercial/Brand (Prefixed)    11259
Mobile Number                    325
Numeric Shortcode                268
Commercial/Brand (Textual)       132
Name: count, dtype: int64 

Unique senders in 'Others': 0


In [7]:
inbox_df[inbox_df['sender_category'] == 'Mobile Number']

,id,date,sender,text,is_from_me,service,sender_category
50,55,2021-05-29 10:00:11.000000,+919351197465,6-Layers Protection C0R0NA-virus.\nwear N95 Ma...,False,SMS,Mobile Number
94,102,2021-06-06 07:17:53.000000,+917703967847,"Dear 96991837XX,\n\nRs.5500 in Your Rummy Acco...",False,SMS,Mobile Number
118,126,2021-06-09 10:02:53.000000,+918700855621,"Congratulations,\n\nRs.5500 in Your Rummy Acco...",False,SMS,Mobile Number
266,269,2021-06-30 05:10:44.000000,+917990025887,Your 9699XXXX27 is selected for Rs 5250.00 dir...,False,SMS,Mobile Number
398,409,2021-07-26 07:06:58.000000,+918595950300,Your Citi Credit Card No. YCPTXXXXXXXX8795 has...,False,SMS,Mobile Number
...,...,...,...,...,...,...,...
12112,17882,2026-02-07 20:33:24.276616,+919981657790,😞,True,SMS,Mobile Number
12113,17883,2026-02-07 20:33:25.558564,+919981657790,😞,True,SMS,Mobile Number
12114,17884,2026-02-07 20:33:47.968934,+919981657790,Tum so hi rhi ho na,True,SMS,Mobile Number
12115,17885,2026-02-07 20:34:13.965093,+919981657790,Kitna piya tha itna,True,SMS,Mobile Number


In [8]:
import re
import pandas as pd
import sys

# 1. Focus only on Commercial and Shortcode messages (Drop personal mobile numbers)
commercial_df = inbox_df[inbox_df['sender_category'] != 'Mobile Number'].copy()

# 2. Define a comprehensive case-insensitive regex pattern for financial keywords
# Handled a/c forms correctly and standard boundaries.
fin_pattern = re.compile(
    r'\b(debited|credited|deducted|received|spent|paid|payment|refunded|'
    r'inr|account|acct|balance|bal|vpa|upi|neft|imps|rtgs|atm|pos|card|txn|transaction)\b|'
    r'a/c|\brs\.?|₹', 
    re.IGNORECASE
)

def has_financial_keyword(text):
    if not isinstance(text, str):
        return False
    return bool(fin_pattern.search(text))

print("Applying financial regex to commercial SMS... (Showing progress)")

results = []
total = len(commercial_df)
for i, text in enumerate(commercial_df['text']):
    results.append(has_financial_keyword(text))
    # Print progress every 2000 rows
    if (i + 1) % 2000 == 0:
        sys.stdout.write(f"\rProcessed {i+1} out of {total} messages...")
        sys.stdout.flush()

sys.stdout.write(f"\rProcessed {total} out of {total} messages... Done!\n")
commercial_df['1_has_fin_keyword'] = results


# 4. Analyze the breakdown
keyword_counts = commercial_df['1_has_fin_keyword'].value_counts()
print(f"\nTotal Commercial/Shortcode messages: {total}")
print("Messages with Financial Keywords vs Without:")
print(keyword_counts)
print(f"Percentage with keywords: {(keyword_counts.get(True, 0) / total) * 100:.2f}%\n")

# 5. Let's inspect a few that MATCHED
print("--- SAMPLE MATCHED MESSAGES ---")
matched_sample = commercial_df[commercial_df['1_has_fin_keyword'] == True]['text'].dropna().sample(5, random_state=42)
for text in matched_sample:
    print(f"- {text[:150].replace(chr(10), ' ')}...")

# 6. Let's inspect a few that UNMATCHED (to see what we are safely ignoring, or if we missed anything)
print("\n--- SAMPLE UNMATCHED MESSAGES ---")
unmatched_sample = commercial_df[commercial_df['1_has_fin_keyword'] == False]['text'].dropna().sample(5, random_state=42)
for text in unmatched_sample:
    print(f"- {text[:150].replace(chr(10), ' ')}...")

Applying financial regex to commercial SMS... (Showing progress)
Processed 11659 out of 11659 messages... Done!

Total Commercial/Shortcode messages: 11659
Messages with Financial Keywords vs Without:
1_has_fin_keyword
True     6624
False    5035
Name: count, dtype: int64
Percentage with keywords: 56.81%

--- SAMPLE MATCHED MESSAGES ---
- Hi Manish, Use Your EyeMyEye Code LAUNCH & Get Sunglasses Starting @Rs. 249 & Eyeglasses Starting @Rs. 249: https://t1x.in/Aq2gtAsDdMR7Gp...
- NEXTBILLION TECHNOLOGY at EOD 02/04/2022 reported your Fund bal Rs 29.430 & Securities bal 0. This excludes your Bank, DP and PMS bal with the broker-...
- Dear CSLXXXXX3D,Your traded value for 19-JUL-22 CM Rs 15231.25 Check your registered email. For details contact broker -National Stock Exchange....
- Signature fragrances from TataNeu! Get up to 65% off on perfumes! Use code NEU500 & get Rs.500 off + 5% NeuCoins! T&C apply resu.io/XMEKN5GES2NZa - Ta...
- Hi Manish, Dominos is LIVE on Tata Neu! Just for you -

In [9]:
import os

# Create RESULTS directory
os.makedirs('RESULTS', exist_ok=True)

# Separate the datasets based on our current heuristic
likely_transactions = commercial_df[commercial_df['1_has_fin_keyword'] == True]
non_transactions = commercial_df[commercial_df['1_has_fin_keyword'] == False]

# Save to a single Excel file with two sheets
output_path = 'RESULTS/1_transactions_split.xlsx'
with pd.ExcelWriter(output_path) as writer:
    likely_transactions.to_excel(writer, sheet_name='likely_transactions', index=False)
    non_transactions.to_excel(writer, sheet_name='non_transactions', index=False)

print(f"Saved {len(likely_transactions)} likely transactions and {len(non_transactions)} non-transactions to {output_path}")

Saved 6624 likely transactions and 5035 non-transactions to RESULTS/1_transactions_split.xlsx


In [10]:
# Let's refine the heuristic to reduce false positives (promotional messages)

# 1. Define a pattern for promotional / non-transactional keywords
promo_pattern = re.compile(
    r'\b(offer|offers|discount|promo|code|flat|off|sale|win|free|cashback|apply|starting @|subscribe|bonus|gift|hurry|rewards?)\b|%', 
    re.IGNORECASE
)

# We can also tighten the financial pattern to require stronger transactional verbs if needed in the future,
# but for now, let's see how much we can clean up just by excluding promotional text.

def has_promo_keyword(text):
    if not isinstance(text, str):
        return False
    
    # Needs to match the base financial keywords
    has_fin = bool(fin_pattern.search(text))
    # MUST NOT look like an obvious promotion
    is_promo = bool(promo_pattern.search(text))
    
    return has_fin and not is_promo

print("Applying tightened filtering to remove promotional noise...")
commercial_df['2_has_promo_keyword'] = commercial_df['text'].apply(has_promo_keyword)

# Analyze refined breakdown
txn_counts = commercial_df['2_has_promo_keyword'].value_counts()
print("\nRefined classification (Likely Transaction vs Not):")
print(txn_counts)
print(f"Percentage classified as actual transaction: {(txn_counts.get(True, 0) / len(commercial_df)) * 100:.2f}%\n")

print("--- SAMPLE REFINED TRANSACTIONS (True) ---")
refined_true = commercial_df[commercial_df['2_has_promo_keyword'] == True]['text'].dropna().sample(5)
for text in refined_true:
    print(f"- {text[:150].replace(chr(10), ' ')}")

print("\n--- SAMPLE FILTERED OUT AS PROMO (Matched fin_keyword initially, but rejected now) ---")
filtered_out = commercial_df[(commercial_df['1_has_fin_keyword'] == True) & (commercial_df['2_has_promo_keyword'] == False)]['text'].dropna().sample(5, random_state=42)
for text in filtered_out:
    print(f"- {text[:150].replace(chr(10), ' ')}")

Applying tightened filtering to remove promotional noise...

Refined classification (Likely Transaction vs Not):
2_has_promo_keyword
False    7273
True     4386
Name: count, dtype: int64
Percentage classified as actual transaction: 37.62%

--- SAMPLE REFINED TRANSACTIONS (True) ---
- विनर बनने का मौका! खेलें कॉन्टेस्ट और जीतें कार@ Rs5/Q.डायल 50404.
- Dispatched: Credit Card Newly issued for ICICI Bank Card XX6009 sent by Blue Dart Courier, AWB 33662269174 on 25-SEP-24. icici.co/DUvfZIqR5ZE .
- Article No:EM971104431IN received @ Ambernath S.O on 23/06/2021 10:33:25.Track @ www.indiapost.gov.in
- HDFC Bank: Your UPI-Mandate is successfully cancelled by  manisharadwad1 for ? 14980.00.
- Dear Customer, get a pre-approved Personal Loan of Rs. 179000 from IDFC FIRST Bank with ZERO hassle & documentation. https://idfcfs.in/DmC3PC TCA

--- SAMPLE FILTERED OUT AS PROMO (Matched fin_keyword initially, but rejected now) ---
- Manish, Best deal alert! Get a Loan of Rs.150000 on your HDFC Bank Cr

In [11]:
import os

# Create RESULTS directory
os.makedirs('RESULTS', exist_ok=True)

# Separate the datasets based on our current heuristic
likely_transactions = commercial_df[commercial_df['2_has_promo_keyword'] == True]
non_transactions = commercial_df[commercial_df['2_has_promo_keyword'] == False]

# Save to a single Excel file with two sheets
output_path = 'RESULTS/2_transactions_split.xlsx'
with pd.ExcelWriter(output_path) as writer:
    likely_transactions.to_excel(writer, sheet_name='likely_transactions', index=False)
    non_transactions.to_excel(writer, sheet_name='non_transactions', index=False)

print(f"Saved {len(likely_transactions)} likely transactions and {len(non_transactions)} non-transactions to {output_path}")

Saved 4386 likely transactions and 7273 non-transactions to RESULTS/2_transactions_split.xlsx
